In [ ]:
"""
ROGII Wellbore Geology Prediction - physics + GR correlation ensemble.

The public test files are label-leaked locally, so this notebook uses the same
general path needed for hidden wells: formation interpolation, conservative
residual decay, GR/typewell NCC diagnostics, and confidence-gated blending.
"""

import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator


LOCAL_DATA_DIR = "/Users/VR/Desktop/docs/playground/data/rogii"
KAGGLE_DATA_DIRS = [
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    "/kaggle/input/rogii-wellbore-geology-prediction",
]
FORM_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
NCC_WINDOWS_FT = [50.0, 100.0, 200.0, 500.0]
NCC_SEARCH_RADIUS_FT = 180.0
NCC_STRIDE_ROWS = 25
DECAY_TAU_MD = 2000.0
USE_NCC = False


def resolve_data_dir() -> str:
    for path in KAGGLE_DATA_DIRS + [LOCAL_DATA_DIR]:
        if os.path.exists(os.path.join(path, "train")) and os.path.exists(os.path.join(path, "test")):
            return path
    raise FileNotFoundError("Could not find ROGII data directory")


def well_id_from_file(filename: str) -> str:
    return filename.replace("__horizontal_well.csv", "")


def first_prediction_index(hw: pd.DataFrame) -> int:
    if hw["TVT_input"].isna().any():
        return int(hw["TVT_input"].isna().idxmax())
    return len(hw)


def rmse(pred: np.ndarray, truth: np.ndarray) -> float:
    mask = np.isfinite(pred) & np.isfinite(truth)
    if not mask.any():
        return float("nan")
    return float(np.sqrt(np.mean((pred[mask] - truth[mask]) ** 2)))


def fill_series(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    y = y.astype(float).copy()
    mask = np.isfinite(y)
    if mask.sum() == 0:
        return y
    if mask.sum() == 1:
        y[~mask] = y[mask][0]
        return y
    y[~mask] = np.interp(x[~mask], x[mask], y[mask])
    return y


def smooth_array(values: np.ndarray, width: int = 9) -> np.ndarray:
    if len(values) < 3 or width <= 1:
        return values
    width = min(width, len(values) if len(values) % 2 == 1 else len(values) - 1)
    if width < 3:
        return values
    pad = width // 2
    kernel = np.ones(width, dtype=float) / width
    padded = np.pad(values, (pad, pad), mode="edge")
    return np.convolve(padded, kernel, mode="valid")


def corrcoef_safe(a: np.ndarray, b: np.ndarray) -> float:
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 10:
        return -1.0
    aa = a[mask] - np.mean(a[mask])
    bb = b[mask] - np.mean(b[mask])
    denom = np.linalg.norm(aa) * np.linalg.norm(bb)
    if denom <= 1e-9:
        return -1.0
    return float(np.dot(aa, bb) / denom)


@dataclass
class PhysicsResult:
    pred: np.ndarray
    tvt_curve_all: np.ndarray
    cal_rmse: float
    formation: str
    tail_residual: float


@dataclass
class NCCResult:
    pred: np.ndarray
    confidence: np.ndarray
    correction: np.ndarray


class FormationInterpolator:
    def __init__(self, data_dir: str, exclude_wells=None, sample_step: int = 20):
        self.data_dir = data_dir
        self.train_dir = os.path.join(data_dir, "train")
        self.exclude_wells = set(exclude_wells or [])
        self.sample_step = sample_step
        self.linear = {}
        self.nearest = {}

    def build(self):
        train_files = sorted(
            f for f in os.listdir(self.train_dir)
            if f.endswith("__horizontal_well.csv") and well_id_from_file(f) not in self.exclude_wells
        )
        print(f"Building formation interpolators from {len(train_files)} wells...")
        for col in FORM_COLS:
            all_xy, all_z = [], []
            for filename in train_files:
                hw = pd.read_csv(os.path.join(self.train_dir, filename), usecols=["X", "Y", col])
                sampled = hw.iloc[:: self.sample_step]
                all_xy.append(sampled[["X", "Y"]].values)
                all_z.append(sampled[col].values)
            xy_arr = np.vstack(all_xy)
            z_arr = np.concatenate(all_z)
            self.linear[col] = LinearNDInterpolator(xy_arr, z_arr)
            self.nearest[col] = NearestNDInterpolator(xy_arr, z_arr)
            print(f"  {col}: {len(xy_arr):,} pts")
        return self

    def interpolate(self, col: str, xy: np.ndarray) -> np.ndarray:
        vals = self.linear[col](xy)
        mask = ~np.isfinite(vals)
        if mask.any():
            vals[mask] = self.nearest[col](xy[mask])
        return vals


def residual_decay(md_known: np.ndarray, residuals: np.ndarray, md_pred: np.ndarray):
    tail_n = min(500, len(md_known))
    tail_md = md_known[-tail_n:]
    tail_res = residuals[-tail_n:]
    valid = np.isfinite(tail_res)
    if valid.sum() < 5:
        intercept = float(np.nanmedian(residuals)) if np.isfinite(residuals).any() else 0.0
        return np.full_like(md_pred, intercept, dtype=float), intercept

    anchor_md = tail_md[-1]
    x_tail = tail_md[valid] - anchor_md
    y_tail = tail_res[valid]
    if valid.sum() >= 10 and np.nanstd(x_tail) > 1e-6:
        slope, intercept = np.polyfit(x_tail, y_tail, 1)
    else:
        slope, intercept = 0.0, float(np.nanmedian(y_tail))

    x_future = md_pred - anchor_md
    decay = np.exp(-np.maximum(x_future, 0.0) / DECAY_TAU_MD)
    correction = intercept + slope * x_future * decay
    correction = np.clip(correction, -75.0, 75.0)
    return correction, float(intercept)


def predict_physics(hw: pd.DataFrame, interpolator: FormationInterpolator, pred_start_idx=None) -> PhysicsResult:
    hw = hw.sort_values("MD").reset_index(drop=True).copy()
    ps_idx = first_prediction_index(hw) if pred_start_idx is None else int(pred_start_idx)
    xy = hw[["X", "Y"]].values
    for col in FORM_COLS:
        hw[f"{col}_i"] = interpolator.interpolate(col, xy)

    known = hw.iloc[:ps_idx]
    pred = hw.iloc[ps_idx:]
    if pred.empty:
        return PhysicsResult(np.array([]), hw["TVT_input"].values, 0.0, FORM_COLS[0], 0.0)
    if known["TVT_input"].notna().sum() < 5:
        fallback = hw["TVT_input"].dropna()
        base_value = float(fallback.iloc[-1]) if len(fallback) else float(pred["Z"].median())
        curve = np.concatenate([hw.iloc[:ps_idx]["TVT_input"].fillna(base_value).values, np.full(len(pred), base_value)])
        return PhysicsResult(np.full(len(pred), base_value), curve, 999.0, "fallback", 0.0)

    best = None
    for col in FORM_COLS:
        fi_known = known[f"{col}_i"].values
        fi_pred = pred[f"{col}_i"].values
        tvt_known = known["TVT_input"].values
        z_known = known["Z"].values
        z_pred = pred["Z"].values
        md_known = known["MD"].values
        md_pred = pred["MD"].values

        implied = tvt_known - (fi_known - z_known)
        valid = np.isfinite(implied) & np.isfinite(tvt_known)
        if valid.sum() < 5:
            continue
        tvt_form = np.median(implied[valid])
        base_known = tvt_form + (fi_known - z_known)
        base_pred = tvt_form + (fi_pred - z_pred)
        residuals = tvt_known - base_known
        score = rmse(base_known, tvt_known)

        pred_corr, tail_res = residual_decay(md_known, residuals, md_pred)
        candidate = base_pred + 0.15 * pred_corr
        if best is None or score < best[0]:
            tvt_curve_all = np.concatenate([tvt_known, candidate])
            best = (score, candidate, tvt_curve_all, col, tail_res)

    if best is None:
        fallback = known["TVT_input"].dropna()
        base_value = float(fallback.iloc[-1]) if len(fallback) else float(pred["Z"].median())
        curve = np.concatenate([known["TVT_input"].fillna(base_value).values, np.full(len(pred), base_value)])
        return PhysicsResult(np.full(len(pred), base_value), curve, 999.0, "fallback", 0.0)

    return PhysicsResult(
        pred=best[1],
        tvt_curve_all=best[2],
        cal_rmse=float(best[0]),
        formation=best[3],
        tail_residual=best[4],
    )


def ncc_for_anchor(hw_all: pd.DataFrame, tvt_curve_all: np.ndarray, typewell: pd.DataFrame, anchor_idx: int, window_ft: float):
    anchor_tvt = tvt_curve_all[anchor_idx]
    half = window_ft / 2.0
    rel_grid = np.linspace(-half, half, max(25, int(window_ft / 2.0) + 1))

    md = hw_all["MD"].values
    gr = fill_series(md, hw_all["GR"].values.astype(float))
    valid_gr = np.isfinite(hw_all["GR"].values.astype(float))

    win_mask = np.abs(tvt_curve_all - anchor_tvt) <= half
    win_mask &= np.isfinite(gr) & np.isfinite(tvt_curve_all)
    coverage = float((win_mask & valid_gr).sum() / max(1, win_mask.sum()))
    if win_mask.sum() < 15 or coverage < 0.35:
        return np.nan, -1.0, coverage

    order = np.argsort(tvt_curve_all[win_mask])
    h_tvt = tvt_curve_all[win_mask][order]
    h_gr = gr[win_mask][order]
    unique_tvt, unique_idx = np.unique(h_tvt, return_index=True)
    if len(unique_tvt) < 10:
        return np.nan, -1.0, coverage
    h_seg = np.interp(anchor_tvt + rel_grid, unique_tvt, h_gr[unique_idx])
    if np.nanstd(h_seg) < 1.0:
        return np.nan, -1.0, coverage

    tw_tvt = typewell["TVT"].values.astype(float)
    tw_gr = fill_series(tw_tvt, typewell["GR"].values.astype(float))
    lo = max(tw_tvt.min() + half, anchor_tvt - NCC_SEARCH_RADIUS_FT)
    hi = min(tw_tvt.max() - half, anchor_tvt + NCC_SEARCH_RADIUS_FT)
    if hi <= lo:
        return np.nan, -1.0, coverage

    centers = np.arange(lo, hi + 0.001, 1.0)
    best_center, best_corr = np.nan, -1.0
    for center in centers:
        tw_seg = np.interp(center + rel_grid, tw_tvt, tw_gr)
        cc = corrcoef_safe(h_seg, tw_seg)
        if cc > best_corr:
            best_corr = cc
            best_center = center
    return best_center, best_corr, coverage


def predict_ncc(hw_sorted: pd.DataFrame, typewell: pd.DataFrame, physics: PhysicsResult, ps_idx: int) -> NCCResult:
    pred_len = len(hw_sorted) - ps_idx
    if pred_len <= 0 or typewell is None or "GR" not in hw_sorted or hw_sorted["GR"].notna().sum() < 50:
        return NCCResult(physics.pred.copy(), np.zeros(pred_len), np.zeros(pred_len))

    anchor_offsets = list(range(0, pred_len, NCC_STRIDE_ROWS))
    if anchor_offsets[-1] != pred_len - 1:
        anchor_offsets.append(pred_len - 1)

    corr_points, conf_points, row_points = [], [], []
    for offset in anchor_offsets:
        idx = ps_idx + offset
        estimates = []
        weights = []
        for window_ft in NCC_WINDOWS_FT:
            center, cc, coverage = ncc_for_anchor(hw_sorted, physics.tvt_curve_all, typewell, idx, window_ft)
            if np.isfinite(center) and cc > 0.15:
                weight = max(0.0, cc - 0.15) * coverage * np.sqrt(window_ft / 100.0)
                estimates.append(center)
                weights.append(weight)
        if weights:
            estimate = float(np.average(estimates, weights=weights))
            correction = np.clip(estimate - physics.tvt_curve_all[idx], -90.0, 90.0)
            confidence = float(np.clip(np.sum(weights) / 4.0, 0.0, 1.0))
        else:
            correction = 0.0
            confidence = 0.0
        row_points.append(offset)
        corr_points.append(correction)
        conf_points.append(confidence)

    x_all = np.arange(pred_len)
    correction = np.interp(x_all, row_points, corr_points)
    confidence = np.interp(x_all, row_points, conf_points)
    correction = smooth_array(correction, width=11)
    confidence = smooth_array(confidence, width=11)
    return NCCResult(physics.pred + correction, confidence, correction)


def blend_predictions(physics: PhysicsResult, ncc: NCCResult, hw_pred: pd.DataFrame) -> np.ndarray:
    gr_coverage = hw_pred["GR"].notna().rolling(101, min_periods=1, center=True).mean().values
    cal_factor = np.clip((physics.cal_rmse - 8.0) / 12.0, 0.0, 1.0)
    conf_factor = np.clip((ncc.confidence - 0.75) / 0.20, 0.0, 1.0)
    weight = np.clip(0.10 * cal_factor * conf_factor * gr_coverage, 0.0, 0.10)
    return (1.0 - weight) * physics.pred + weight * ncc.pred


def load_typewell(data_dir: str, split: str, well_id: str):
    path = os.path.join(data_dir, split, f"{well_id}__typewell.csv")
    if not os.path.exists(path):
        return None
    tw = pd.read_csv(path)
    return tw.sort_values("TVT").reset_index(drop=True)


def predict_well(data_dir: str, split: str, well_id: str, hw: pd.DataFrame, interpolator: FormationInterpolator, pred_start_idx=None):
    hw_sorted = hw.sort_values("MD").reset_index(drop=True)
    ps_idx = first_prediction_index(hw_sorted) if pred_start_idx is None else int(pred_start_idx)
    physics = predict_physics(hw_sorted, interpolator, ps_idx)
    if USE_NCC:
        typewell = load_typewell(data_dir, split, well_id)
        ncc = predict_ncc(hw_sorted, typewell, physics, ps_idx)
    else:
        ncc = NCCResult(physics.pred.copy(), np.zeros(len(physics.pred)), np.zeros(len(physics.pred)))
    final = blend_predictions(physics, ncc, hw_sorted.iloc[ps_idx:])
    print(
        f"  {well_id}: formation={physics.formation} cal={physics.cal_rmse:.3f} "
        f"ncc_conf={np.nanmean(ncc.confidence):.3f}"
    )
    return final, physics, ncc


def make_submission(data_dir: str):
    test_dir = os.path.join(data_dir, "test")
    sample_path = os.path.join(data_dir, "sample_submission.csv")
    test_files = sorted(f for f in os.listdir(test_dir) if f.endswith("__horizontal_well.csv"))
    interpolator = FormationInterpolator(data_dir).build()
    sample = pd.read_csv(sample_path) if os.path.exists(sample_path) else None

    all_ids, all_tvts = [], []
    for filename in test_files:
        well_id = well_id_from_file(filename)
        hw = pd.read_csv(os.path.join(test_dir, filename))
        hw_with_index = hw.sort_values("MD").reset_index(drop=False)
        if sample is not None:
            prefix = f"{well_id}_"
            needed = sample[sample["id"].str.startswith(prefix)]["id"].str[len(prefix):].astype(int).values
            if len(needed) == 0:
                continue
            needed_set = set(needed.tolist())
            target_positions = np.flatnonzero(hw_with_index["index"].isin(needed_set).values)
            pred_start = int(target_positions.min()) if len(target_positions) else first_prediction_index(hw_with_index)
        else:
            pred_start = first_prediction_index(hw_with_index)
            needed = hw_with_index.iloc[pred_start:]["index"].values
            needed_set = set(needed.tolist())

        pred, _, _ = predict_well(data_dir, "test", well_id, hw_with_index, interpolator, pred_start)
        pred_rows = hw_with_index.iloc[pred_start:].copy()
        pred_rows["tvt"] = pred
        pred_rows = pred_rows[pred_rows["index"].isin(needed_set)]
        all_ids.extend([f"{well_id}_{idx}" for idx in pred_rows["index"].values])
        all_tvts.extend(pred_rows["tvt"].tolist())

    submission = pd.DataFrame({"id": all_ids, "tvt": all_tvts})
    if sample is not None:
        submission = sample[["id"]].merge(submission, on="id", how="left")
        if submission["tvt"].isna().any():
            submission["tvt"] = submission["tvt"].interpolate(limit_direction="both")
            submission["tvt"] = submission["tvt"].fillna(submission["tvt"].median())

    out_path = "submission.csv" if data_dir.startswith("/kaggle/") else os.path.join(data_dir, "submission.csv")
    submission.to_csv(out_path, index=False)
    print(f"Saved {out_path}: shape={submission.shape}, tvt=({submission.tvt.min():.2f}, {submission.tvt.max():.2f})")
    return submission


data_dir = resolve_data_dir()
print(f"Using data: {data_dir}")
make_submission(data_dir)